# Contextual DDM with a design matrix

This example regresses DDM drift rate (`v`) and non-decision time (`tau`) on trial-level covariates. `DesignMatrix` turns the formulas into vectorized NumPy operations before the DDM is simulated.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import superstats as sup
from superstats.transition import RandomWalk

## Simulate trial context

The context simulator returns one covariate sequence per simulated dataset. In an application, this function can instead draw from an experimental design or resample observed trial metadata.

In [ ]:
def simulate_context(*, batch_size, num_steps):
    cue_validity = np.broadcast_to(np.linspace(-1.0, 1.0, num_steps), (batch_size, num_steps))
    n_cues = np.broadcast_to(1.0 + (np.arange(num_steps) % 3), (batch_size, num_steps))
    return {"cue_validity": cue_validity, "n_cues": n_cues}

context = sup.ContextSimulator(simulate_context)

## Specify regressions and the DDM

In [ ]:
design = sup.DesignMatrix([
    "v = v_0 + b_v * cue_validity",
    "tau = tau_0 + b_tau * n_cues",
])

prior = sup.JointPrior(
    v_0=sup.Prior("normal", loc=0.0, scale=2),
    b_v=RandomWalk(bounds=(-4.0, 4.0), initial_prior=sup.Prior("normal", loc=0.0, scale=0.5), sigma=sup.Prior("halfnormal", loc=0.0, scale=0.1)),
    tau_0=RandomWalk(bounds=(0.1, 0.8), initial_prior=sup.Prior("normal", loc=0.35, scale=0.03), sigma=sup.Prior("halfnormal", loc=0.0, scale=0.1)),
    b_tau=sup.Prior("normal", loc=0.0, scale=0.5),
    a=1.5,
    bias=0.5,
)

model = sup.Model(
    prior=prior,
    simulator=sup.simulation.sample_ddm,
    missing=None,
    context=context,
    context_mapping=sup.ContextMapping(design_context=("cue_validity", "n_cues")),
    design_matrix=design,
)

## Sample and inspect

The sampled output retains latent baseline parameters and context. The resolved DDM parameters below are reconstructed with the same formulas for visualization.

In [ ]:
sample = model.sample(batch_size=3, num_steps=75)

v = sample["v_0"][:, None, :] + sample["b_v"] * sample["cue_validity"][..., None]
tau = sample["tau_0"] + sample["b_tau"][:, None, :] * sample["n_cues"][..., None]

fig, axes = plt.subplots(3, 1, figsize=(9, 7), sharex=True)
axes[0].plot(sample["cue_validity"][0], label="cue validity")
axes[0].plot(sample["n_cues"][0], label="number of cues")
axes[0].legend()
axes[0].set_ylabel("context")
axes[1].plot(v[0, :, 0], label="drift rate v")
axes[1].plot(tau[0, :, 0], label="non-decision time tau")
axes[1].legend()
axes[1].set_ylabel("resolved parameter")
axes[2].scatter(sample["time_steps"][0], sample["response_time"][0], c=sample["choice"][0], cmap="coolwarm", s=18)
axes[2].set(xlabel="trial", ylabel="response time")
fig.tight_layout()